# 🧬 03 — Metabolic Stacked LSTM · Chapter 5 SF-07 / LF-2.2
**TwinPacemaker Digital Twin Framework — Université Constantine 2**

| Parameter | Value |
|---|---|
| **Dataset** | GlucoBench (Kaggle: `omenkj/glucobench-glucose-monitoring-and-lifestyle-data`) |
| **Window** | 120 readings × 5 min = **10 hours** lookback |
| **Horizon** | 12 readings × 5 min = **60 minutes** ahead |
| **Features** | glucose, heart_rate, skin_temp / activity proxy, carbs |
| **Target (thesis)** | MAE < 15 mg/dL · RMSE < 20 mg/dL · R² > 0.92 |
| **Outputs** | `metabolic_stacked_lstm.keras`, `metabolic_model_info.json`, `metabolic_normalization_stats.json`, plots |

> **SF-07** — *Simulate_Metabolic_Behavior*: maintain a virtual metabolic replica synchronised with CGM data  
> **LF-2.2** — *GRU/LSTM Simulation Métabolique*: features `[glucose, HR, temp, activity]`, architecture Stacked LSTM → replaces Bergman synthetic data with real GlucoBench wearable data

## 📦 Step 0 — Install dependencies & mount Google Drive

In [ ]:
# ─── Install / verify core packages ───────────────────────────────────────────
!pip install -q kaggle tensorflow scikit-learn matplotlib pandas numpy

import os, sys, json, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow  : {tf.__version__}")
print(f"GPUs found  : {tf.config.list_physical_devices('GPU')}")
print(f"Python      : {sys.version.split()[0]}")

In [ ]:
# ─── Mount Google Drive ────────────────────────────────────────────────────────
# Your dataset zip should be placed at:
#   MyDrive/TwinPacemaker/data/glucobench.zip
# OR the extracted CSV at:
#   MyDrive/TwinPacemaker/data/glucobench/unified_wearable_glycemia.csv

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Paths (adjust DRIVE_PROJECT_ROOT if your folder is named differently) ──
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/TwinPacemaker')
DRIVE_DATA_DIR     = DRIVE_PROJECT_ROOT / 'data'
DRIVE_MODELS_DIR   = DRIVE_PROJECT_ROOT / 'models' / 'metabolic'

# Local fast-access scratch (Colab /content)
LOCAL_DATA_DIR   = Path('/content/glucobench_data')
LOCAL_MODELS_DIR = Path('/content/models/metabolic')

for d in [DRIVE_DATA_DIR, DRIVE_MODELS_DIR, LOCAL_DATA_DIR, LOCAL_MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive project root : {DRIVE_PROJECT_ROOT}")
print(f"Local scratch      : {LOCAL_DATA_DIR}")
print(f"Models saved to    : {DRIVE_MODELS_DIR}  (syncs to your D:\\Vibe Coding\\TwinPacemaker\\models\\metabolic)")

## 📥 Step 1 — Load GlucoBench dataset from Drive

**Two options** (the cell auto-detects which one you have):
- **Option A** — You already extracted the CSV and placed it in Drive
- **Option B** — You downloaded the Kaggle ZIP and placed it in Drive
- **Option C** — Download directly from Kaggle API (requires `kaggle.json` in Drive)

In [ ]:
# ─── Dataset discovery & loading ──────────────────────────────────────────────
# Known GlucoBench CSV filenames (the dataset may unzip under different names)
CANDIDATE_CSV_NAMES = [
    'unified_wearable_glycemia.csv',
    'glucobench.csv',
    'glucose_data.csv',
    'data.csv',
]

def find_csv(search_dirs):
    """Search multiple directories for the GlucoBench CSV."""
    for d in search_dirs:
        d = Path(d)
        for name in CANDIDATE_CSV_NAMES:
            p = d / name
            if p.exists():
                return p
        # Deep search up to 2 levels
        for f in d.rglob('*.csv'):
            if any(kw in f.name.lower() for kw in ['glucose', 'glycemia', 'glucobench', 'cgm']):
                return f
    return None

DATA_CSV = find_csv([DRIVE_DATA_DIR, LOCAL_DATA_DIR])

# ── Option C: Download from Kaggle if not found ─────────────────────────────
if DATA_CSV is None:
    print("CSV not found in Drive. Attempting Kaggle download...")
    kaggle_json = DRIVE_PROJECT_ROOT / 'kaggle.json'
    if kaggle_json.exists():
        os.makedirs('/root/.config/kaggle', exist_ok=True)
        shutil.copy(kaggle_json, '/root/.config/kaggle/kaggle.json')
        os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
        !kaggle datasets download -d omenkj/glucobench-glucose-monitoring-and-lifestyle-data \
            -p /content/glucobench_data --unzip -q
        DATA_CSV = find_csv([LOCAL_DATA_DIR])
    else:
        raise FileNotFoundError(
            "\n" + "="*60 +
            "\nCSV not found! Please do ONE of the following:"
            "\n  1. Place your extracted CSV in: " + str(DRIVE_DATA_DIR) +
            "\n  2. Place glucobench.zip    in: " + str(DRIVE_DATA_DIR) +
            "\n  3. Place kaggle.json       in: " + str(DRIVE_PROJECT_ROOT) +
            "\n" + "="*60
        )

# ── Option B: Unzip from Drive ───────────────────────────────────────────────
if DATA_CSV is None:
    zips = list(DRIVE_DATA_DIR.glob('*.zip'))
    if zips:
        print(f"Extracting {zips[0].name} ...")
        import zipfile
        with zipfile.ZipFile(zips[0], 'r') as zf:
            zf.extractall(LOCAL_DATA_DIR)
        DATA_CSV = find_csv([LOCAL_DATA_DIR])

if DATA_CSV is None:
    raise FileNotFoundError("Could not locate GlucoBench CSV. Check paths above.")

print(f"\n✅ Dataset found: {DATA_CSV}")
print(f"   Size: {DATA_CSV.stat().st_size / 1e6:.1f} MB")

## 🔬 Step 2 — Preprocessing (multi-patient, clinical imputation)

In [ ]:
# ─── Hyperparameters ──────────────────────────────────────────────────────────
WINDOW_SIZE    = 120   # 10 hours lookback (matches LF-2.2 Input(120,4))
HORIZON_STEPS  = 12    # 60-min horizon → single absolute target at step+12
RESAMPLE_RULE  = '5min'
TRAIN_FRACTION = 0.80
EPOCHS         = 80
BATCH_SIZE     = 256
PATIENCE_ES    = 15
PATIENCE_LR    = 8

# Thesis performance targets (Chapter 5 § 5.6.3)
TARGET_MAE   = 15.0    # mg/dL
TARGET_RMSE  = 20.0    # mg/dL
TARGET_R2    = 0.92

print("═"*60)
print(" TwinPacemaker Metabolic LSTM — SF-07 / LF-2.2")
print("═"*60)
print(f" Window : {WINDOW_SIZE} steps × 5 min = {WINDOW_SIZE*5//60}h lookback")
print(f" Horizon: {HORIZON_STEPS} steps × 5 min = {HORIZON_STEPS*5} min ahead")
print(f" Targets: MAE < {TARGET_MAE} | RMSE < {TARGET_RMSE} | R² > {TARGET_R2}")
print("═"*60)

In [ ]:
# ─── Load raw data ────────────────────────────────────────────────────────────
print("[1/6] Loading raw CSV ...")
raw = pd.read_csv(DATA_CSV)
print(f"  Shape : {raw.shape}")
print(f"  Cols  : {raw.columns.tolist()}")
raw.head(3)

In [ ]:
# ─── Column mapping (robust auto-detection) ───────────────────────────────────
cols_lower = {c.lower(): c for c in raw.columns}

def detect_col(candidates):
    for k in candidates:
        if k in cols_lower:
            return cols_lower[k]
    return None

TIME_COL     = detect_col(['timestamp', 'time', 'datetime', 'date'])
GLUCOSE_COL  = detect_col(['glucose', 'cgm', 'bg', 'sensor_glucose', 'blood_glucose', 'cbg'])
HR_COL       = detect_col(['heart_rate', 'hr', 'heartrate', 'pulse'])
TEMP_COL     = detect_col(['skin_temp', 'temperature', 'temp', 'skin_temperature'])
CARBS_COL    = detect_col(['carbs', 'carbohydrates', 'cho'])
ACTIVITY_COL = detect_col(['exercise_steps', 'steps', 'activity', 'exercise', 'step_count'])
INSULIN_COL  = detect_col(['insulin_bolus', 'insulin', 'bolus'])
USER_COL     = detect_col(['user_id', 'patient_id', 'subject_id', 'id', 'subject'])

print(f"  time     → {TIME_COL}")
print(f"  glucose  → {GLUCOSE_COL}")
print(f"  HR       → {HR_COL}")
print(f"  temp     → {TEMP_COL}")
print(f"  carbs    → {CARBS_COL}")
print(f"  activity → {ACTIVITY_COL}")
print(f"  insulin  → {INSULIN_COL}")
print(f"  user     → {USER_COL}")

if GLUCOSE_COL is None:
    raise ValueError(f"No glucose column detected. Columns: {raw.columns.tolist()}")

# ─── Build feature list (aligned with thesis LF-2.2: glucose, HR, temp, activity)
FEATURE_COLS = [GLUCOSE_COL]
for col, name in [(HR_COL, 'HR'), (TEMP_COL, 'Temp'), (ACTIVITY_COL, 'Activity'), 
                  (CARBS_COL, 'Carbs'), (INSULIN_COL, 'Insulin')]:
    if col:
        FEATURE_COLS.append(col)
        print(f"  ✓ Including {name}: {col}")
    else:
        print(f"  ✗ {name}: not found (will be omitted)")

# Deduplicate
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))
FEATURE_DIM  = len(FEATURE_COLS)
print(f"\n  Final feature vector dim: {FEATURE_DIM}")
print(f"  Features: {FEATURE_COLS}")

In [ ]:
# ─── Per-patient clinical preprocessing ──────────────────────────────────────
print("[2/6] Clinical preprocessing per patient block ...")

raw[TIME_COL] = pd.to_datetime(raw[TIME_COL])

# Event columns → missing = event didn't happen → fill 0
EVENT_COLS = [c for c in [CARBS_COL, INSULIN_COL, ACTIVITY_COL] if c]
for col in EVENT_COLS:
    raw[col] = raw[col].fillna(0)

# Drop CGM quality flags
quality_col = detect_col(['cgm_quality_flag', 'quality_flag', 'quality'])
if quality_col:
    raw.loc[raw[quality_col] == 0, GLUCOSE_COL] = np.nan

processed = []

def process_patient(grp):
    grp = grp.copy().sort_values(TIME_COL).set_index(TIME_COL)
    # Resample to strict 5-min grid
    grp = grp.resample(RESAMPLE_RULE).mean(numeric_only=True)
    # Physiological continuity: linear interpolation
    for col in FEATURE_COLS:
        if col in grp.columns:
            grp[col] = (grp[col]
                        .interpolate(method='time')
                        .ffill().bfill())
    # Physiological range filter
    grp = grp[grp[GLUCOSE_COL].between(40, 400)]
    # Final NaN guard on glucose
    grp = grp.dropna(subset=[GLUCOSE_COL])
    return grp.reset_index()

if USER_COL and raw[USER_COL].nunique() > 1:
    for uid, grp in raw.groupby(USER_COL):
        g = process_patient(grp)
        if len(g) >= (WINDOW_SIZE + HORIZON_STEPS):
            processed.append(g)
    print(f"  Patients processed: {len(processed)}")
else:
    g = process_patient(raw)
    processed.append(g)
    print("  Single-patient dataset processed.")

df = pd.concat(processed, ignore_index=True)
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0)  # Final safety guard

# CGM stats
g_vals = df[GLUCOSE_COL]
hypo   = (g_vals < 70).mean() * 100
hyper  = (g_vals > 180).mean() * 100
print(f"  Total rows : {len(df):,}")
print(f"  Glucose    : {g_vals.mean():.1f} ± {g_vals.std():.1f} mg/dL")
print(f"  Hypo {hypo:.1f}% | Hyper {hyper:.1f}% | Normal {100-hypo-hyper:.1f}%")

In [ ]:
# ─── Sequence tensor generation ───────────────────────────────────────────────
print("[3/6] Generating sliding-window tensors ...")

X_list, y_delta_list, y_abs_list, baseline_list = [], [], [], []

group_col = USER_COL if (USER_COL and df[USER_COL].nunique() > 1 if USER_COL in df.columns else False) else None

def extract_sequences(grp_df):
    features = grp_df[FEATURE_COLS].values.astype(np.float32)
    glucose  = grp_df[GLUCOSE_COL].values.astype(np.float32)
    N = len(grp_df)
    for i in range(N - WINDOW_SIZE - HORIZON_STEPS + 1):
        baseline    = glucose[i + WINDOW_SIZE - 1]       # last CGM in window
        future_abs  = glucose[i + WINDOW_SIZE + HORIZON_STEPS - 1]  # target
        if not (40 <= baseline <= 400 and 40 <= future_abs <= 400):
            continue
        X_list.append(features[i : i + WINDOW_SIZE])
        y_delta_list.append(future_abs - baseline)   # delta target
        y_abs_list.append(future_abs)
        baseline_list.append(baseline)

if group_col and group_col in df.columns:
    for _, grp in df.groupby(group_col):
        extract_sequences(grp)
else:
    extract_sequences(df)

X          = np.array(X_list,       dtype=np.float32)
y_delta    = np.array(y_delta_list, dtype=np.float32)
y_abs      = np.array(y_abs_list,   dtype=np.float32)
baselines  = np.array(baseline_list,dtype=np.float32)

print(f"  Sequences  : {len(y_delta):,}")
print(f"  X shape    : {X.shape}   (samples, window, features)")
print(f"  Delta stats: mean={y_delta.mean():.2f} std={y_delta.std():.2f} mg/dL")

In [ ]:
# ─── Chronological train / test split ─────────────────────────────────────────
split = int(len(y_delta) * TRAIN_FRACTION)
X_train, X_test       = X[:split],        X[split:]
y_train, y_test       = y_delta[:split],  y_delta[split:]
y_abs_train, y_abs_test = y_abs[:split],  y_abs[split:]
base_train, base_test = baselines[:split], baselines[split:]

# ─── Per-feature StandardScaler (fit on train only) ──────────────────────────
print("[4/6] Scaling features ...")
scaler = StandardScaler()
flat_train = X_train.reshape(-1, FEATURE_DIM)
scaler.fit(flat_train)

X_train_sc = scaler.transform(X_train.reshape(-1, FEATURE_DIM)).reshape(len(X_train), WINDOW_SIZE, FEATURE_DIM)
X_test_sc  = scaler.transform(X_test.reshape(-1, FEATURE_DIM)).reshape(len(X_test),  WINDOW_SIZE, FEATURE_DIM)

# Save normalization stats for inference pipeline
g_idx = FEATURE_COLS.index(GLUCOSE_COL)
norm_stats = {
    'scaler': 'StandardScaler',
    'feature_columns': FEATURE_COLS,
    'feature_dim': FEATURE_DIM,
    'glucose_col_index': g_idx,
    'window_size': WINDOW_SIZE,
    'horizon_steps': HORIZON_STEPS,
    'horizon_minutes': HORIZON_STEPS * 5,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'target': {
        'name': 'delta_glucose_mgdl',
        'definition': 'future_glucose[t+12] - last_glucose_in_window',
        'reconstruction': 'predicted_abs = last_window_glucose + predicted_delta'
    },
    'alert_thresholds_mgdl': {'hypoglycemia_lt': 70, 'hyperglycemia_gt': 180},
    'split': {'strategy': 'chronological', 'train_fraction': TRAIN_FRACTION},
    'generated_at_utc': datetime.now(timezone.utc).isoformat()
}

stats_path = LOCAL_MODELS_DIR / 'metabolic_normalization_stats.json'
with open(stats_path, 'w') as f:
    json.dump(norm_stats, f, indent=2)

print(f"  Train: {X_train_sc.shape} | Test: {X_test_sc.shape}")
print(f"  Normalization stats saved → {stats_path.name}")

## 🏗️ Step 3 — Model architecture (Stacked LSTM · SF-07 / LF-2.2)

In [ ]:
# ─── Stacked LSTM — optimised for thesis targets ──────────────────────────────
# Architecture mirrors LF-2.2 (GRU-based) but implemented as deeper LSTM:
# Input(WINDOW, FEATURE_DIM) → LSTM(128) → LSTM(64) → LSTM(32) → Dense(32) → Dense(1)

def build_twinpacemaker_metabolic_lstm(window, feat_dim):
    inp = layers.Input(shape=(window, feat_dim), name='metabolic_sequence')

    # Block 1 — high-capacity feature extraction
    x = layers.LSTM(128, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(1e-4),
                    name='lstm_1')(inp)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Dropout(0.25, name='drop_1')(x)

    # Block 2 — temporal pattern compression
    x = layers.LSTM(64, return_sequences=True,
                    kernel_regularizer=keras.regularizers.l2(1e-4),
                    name='lstm_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Dropout(0.25, name='drop_2')(x)

    # Block 3 — sequence to vector
    x = layers.LSTM(32, return_sequences=False,
                    kernel_regularizer=keras.regularizers.l2(1e-4),
                    name='lstm_3')(x)
    x = layers.BatchNormalization(name='bn_3')(x)
    x = layers.Dropout(0.15, name='drop_3')(x)

    # Dense projection
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4),
                     name='dense_proj')(x)
    x = layers.Dropout(0.10, name='drop_4')(x)

    # Output: predicted glucose DELTA (mg/dL)
    out = layers.Dense(1, activation='linear', name='delta_glucose_output')(x)

    model = Model(inp, out, name='TwinPacemaker_Metabolic_LSTM_SF07')

    # Cosine-annealed learning rate
    lr_sched = keras.optimizers.schedules.CosineDecayRestarts(
        initial_learning_rate=1e-3,
        first_decay_steps=3000,
        t_mul=2.0, m_mul=0.9
    )
    model.compile(
        optimizer=keras.optimizers.Adam(lr_sched),
        loss='huber',       # robust to outlier glucose spikes
        metrics=['mae']
    )
    return model

model = build_twinpacemaker_metabolic_lstm(WINDOW_SIZE, FEATURE_DIM)
model.summary()
print(f"\n  Total parameters: {model.count_params():,}")

## 🚀 Step 4 — Training

In [ ]:
# ─── Callbacks ────────────────────────────────────────────────────────────────
best_model_path = str(LOCAL_MODELS_DIR / 'metabolic_stacked_lstm_best.keras')

cb = [
    callbacks.EarlyStopping(
        monitor='val_mae', patience=PATIENCE_ES,
        restore_best_weights=True, mode='min', verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_mae', factor=0.5,
        patience=PATIENCE_LR, min_lr=1e-7, verbose=1
    ),
    callbacks.ModelCheckpoint(
        best_model_path, monitor='val_mae',
        save_best_only=True, mode='min', verbose=0
    ),
]

print("[5/6] Starting training ...")
print(f"  Epochs={EPOCHS}  Batch={BATCH_SIZE}  Patience={PATIENCE_ES}")

history = model.fit(
    X_train_sc, y_train,
    validation_data=(X_test_sc, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=cb,
    verbose=1
)

best_val_mae = min(history.history['val_mae'])
print(f"\n  ✅ Training complete. Best val MAE (delta): {best_val_mae:.3f} mg/dL")

## 📊 Step 5 — Evaluation (clinical absolute mg/dL)

In [ ]:
# ─── Predict & reconstruct absolute glucose ───────────────────────────────────
print("[6/6] Evaluating on test set ...")

y_pred_delta = model.predict(X_test_sc, verbose=0, batch_size=512).flatten()

# Reconstruct absolute glucose for clinical evaluation
y_pred_abs  = base_test + y_pred_delta
y_true_abs  = y_abs_test

# ── Clinical metrics ──────────────────────────────────────────────────────────
mae   = float(mean_absolute_error(y_true_abs, y_pred_abs))
rmse  = float(np.sqrt(mean_squared_error(y_true_abs, y_pred_abs)))
r2    = float(r2_score(y_true_abs, y_pred_abs))
mape  = float(np.mean(np.abs(y_true_abs - y_pred_abs) / y_true_abs) * 100)

within_15 = float(np.mean(np.abs(y_true_abs - y_pred_abs) <= 15) * 100)
within_20 = float(np.mean(np.abs(y_true_abs - y_pred_abs) <= 20) * 100)

rmse_gl   = rmse / 100  # convert to g/L (thesis comparison)

# ── Alert classification ──────────────────────────────────────────────────────
def classify_zone(v):
    return np.where(v < 70, 'Hypo', np.where(v > 180, 'Hyper', 'Normal'))

true_zone = classify_zone(y_true_abs)
pred_zone = classify_zone(y_pred_abs)
zone_acc  = float(np.mean(true_zone == pred_zone) * 100)

def binary_metrics(true_mask, pred_mask):
    tp = np.sum(true_mask & pred_mask)
    fp = np.sum(~true_mask & pred_mask)
    fn = np.sum(true_mask & ~pred_mask)
    tn = np.sum(~true_mask & ~pred_mask)
    prec = tp/(tp+fp) if (tp+fp) else 0
    rec  = tp/(tp+fn) if (tp+fn) else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    spec = tn/(tn+fp) if (tn+fp) else 0
    return {'precision': float(prec), 'recall': float(rec),
            'f1': float(f1), 'specificity': float(spec)}

hypo_m  = binary_metrics(y_true_abs < 70,  y_pred_abs < 70)
hyper_m = binary_metrics(y_true_abs > 180, y_pred_abs > 180)

# ── Print results ─────────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("        TwinPacemaker METABOLIC MODEL — RESULTS          ")
print("═"*60)
print(f"  MAE            : {mae:.2f} mg/dL   (Target < {TARGET_MAE})  {'✅' if mae < TARGET_MAE else '⚠️'}")
print(f"  RMSE           : {rmse:.2f} mg/dL  (Target < {TARGET_RMSE}) {'✅' if rmse < TARGET_RMSE else '⚠️'}")
print(f"  R²             : {r2:.4f}       (Target > {TARGET_R2}) {'✅' if r2 > TARGET_R2 else '⚠️'}")
print(f"  MAPE           : {mape:.2f}%")
print(f"  RMSE (g/L)     : {rmse_gl:.4f}  (Thesis LF-2.2 benchmark: 0.12 g/L)")
print(f"  Within ±15 mgdL: {within_15:.1f}%")
print(f"  Within ±20 mgdL: {within_20:.1f}%")
print(f"  Zone accuracy  : {zone_acc:.1f}%")
print(f"  Hypo  F1       : {hypo_m['f1']:.3f}  | Recall {hypo_m['recall']:.3f}")
print(f"  Hyper F1       : {hyper_m['f1']:.3f}  | Recall {hyper_m['recall']:.3f}")
all_met = mae < TARGET_MAE and rmse < TARGET_RMSE and r2 > TARGET_R2
print("─"*60)
print(f"  ALL THESIS TARGETS MET: {'✅ YES' if all_met else '⚠️  CHECK ABOVE'}")
print("═"*60)

## 📈 Step 6 — Visualisation
### 6A — Combined dashboard (use in thesis)
All plots in one figure.

In [ ]:
# ─── Shared style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

HYPO_COLOR  = '#e74c3c'
HYPER_COLOR = '#9b59b6'
NORMAL_COLOR= '#27ae60'
PRED_COLOR  = '#2980b9'
TRUE_COLOR  = '#e67e22'

errors = y_true_abs - y_pred_abs

# ─── 6A: 3×3 dashboard ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'TwinPacemaker Metabolic LSTM — SF-07 / LF-2.2\n'
    f'MAE={mae:.1f} mg/dL  RMSE={rmse:.1f} mg/dL  R²={r2:.4f}  Within±15={within_15:.0f}%',
    fontsize=14, fontweight='bold', y=1.01
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ─ Plot 1: Training MAE curve ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history.history['mae'],     label='Train', color='steelblue', lw=1.5)
ax1.plot(history.history['val_mae'], label='Val',   color='darkorange', lw=1.5)
ax1.axhline(TARGET_MAE, color=HYPO_COLOR, ls='--', lw=1.2, label=f'Target {TARGET_MAE}')
ax1.set_title('Training MAE (delta glucose)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MAE (mg/dL)')
ax1.legend(fontsize=9)

# ─ Plot 2: Training Loss curve ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history.history['loss'],     label='Train', color='steelblue', lw=1.5)
ax2.plot(history.history['val_loss'], label='Val',   color='darkorange', lw=1.5)
ax2.set_title('Training Loss (Huber)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(fontsize=9)

# ─ Plot 3: Scatter Predicted vs Actual ────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(y_true_abs, y_pred_abs, s=2, alpha=0.25, color=PRED_COLOR)
lims = [35, 420]
ax3.plot(lims, lims, 'k--', lw=1.5, label='Perfect')
# Zone boundaries
ax3.axhline(70,  color=HYPO_COLOR,  ls=':', lw=1)
ax3.axhline(180, color=HYPER_COLOR, ls=':', lw=1)
ax3.axvline(70,  color=HYPO_COLOR,  ls=':', lw=1)
ax3.axvline(180, color=HYPER_COLOR, ls=':', lw=1)
ax3.set_xlim(lims); ax3.set_ylim(lims)
ax3.set_title(f'Predicted vs Actual (R²={r2:.4f})')
ax3.set_xlabel('Actual (mg/dL)'); ax3.set_ylabel('Predicted (mg/dL)')
ax3.legend(fontsize=9)

# ─ Plot 4: Error distribution ─────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(errors, bins=80, color=PRED_COLOR, edgecolor='white', alpha=0.85)
ax4.axvline(0,   color='black', lw=1.5)
ax4.axvline(-15, color=HYPO_COLOR, ls='--', lw=1.2, label='±15 mg/dL')
ax4.axvline(+15, color=HYPO_COLOR, ls='--', lw=1.2)
ax4.set_title(f'Error Distribution (MAE={mae:.1f})')
ax4.set_xlabel('True − Predicted (mg/dL)'); ax4.set_ylabel('Count')
ax4.legend(fontsize=9)

# ─ Plot 5: Prediction timeline (500 samples) ──────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1:])
N_SHOW = min(500, len(y_true_abs))
t = np.arange(N_SHOW) * 5  # minutes
ax5.plot(t, y_true_abs[:N_SHOW], color=TRUE_COLOR, lw=1.2, label='Actual', alpha=0.9)
ax5.plot(t, y_pred_abs[:N_SHOW], color=PRED_COLOR, lw=1.2, label='Predicted', alpha=0.9, ls='--')
ax5.fill_between(t, 40, 70,  alpha=0.08, color=HYPO_COLOR,  label='Hypoglycemia')
ax5.fill_between(t, 180, 420, alpha=0.08, color=HYPER_COLOR, label='Hyperglycemia')
ax5.set_title(f'60-min Glucose Prediction Timeline (first {N_SHOW} test samples)')
ax5.set_xlabel('Time (minutes)'); ax5.set_ylabel('Glucose (mg/dL)')
ax5.legend(fontsize=9, ncol=2)

# ─ Plot 6: Absolute error over glucose range ─────────────────────────────────
ax6 = fig.add_subplot(gs[2, 0])
ax6.scatter(y_true_abs, np.abs(errors), s=2, alpha=0.2, color='darkorange')
ax6.axhline(TARGET_MAE, color=HYPO_COLOR, ls='--', lw=1.2, label=f'Target {TARGET_MAE} mg/dL')
ax6.axvline(70,  color=HYPO_COLOR,  ls=':', lw=1)
ax6.axvline(180, color=HYPER_COLOR, ls=':', lw=1)
ax6.set_title('Absolute Error vs Glucose Level')
ax6.set_xlabel('True Glucose (mg/dL)'); ax6.set_ylabel('|Error| (mg/dL)')
ax6.legend(fontsize=9)

# ─ Plot 7: Glucose zone bar chart ─────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 1])
zones = ['Hypo\n(<70)', 'Normal\n(70–180)', 'Hyper\n(>180)']
true_counts = [
    np.sum(y_true_abs < 70),
    np.sum((y_true_abs >= 70) & (y_true_abs <= 180)),
    np.sum(y_true_abs > 180)
]
pred_counts = [
    np.sum(y_pred_abs < 70),
    np.sum((y_pred_abs >= 70) & (y_pred_abs <= 180)),
    np.sum(y_pred_abs > 180)
]
x_pos = np.arange(len(zones))
ax7.bar(x_pos - 0.2, true_counts, 0.38, label='Actual',    color=TRUE_COLOR, alpha=0.8)
ax7.bar(x_pos + 0.2, pred_counts, 0.38, label='Predicted', color=PRED_COLOR, alpha=0.8)
ax7.set_xticks(x_pos); ax7.set_xticklabels(zones)
ax7.set_title(f'Glycemic Zone Classification (Acc={zone_acc:.1f}%)')
ax7.set_ylabel('Count')
ax7.legend(fontsize=9)

# ─ Plot 8: Metrics summary bar ────────────────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 2])
metric_names  = ['MAE', 'RMSE', 'R²×100', '±15%', '±20%']
achieved_vals = [mae, rmse, r2*100, within_15, within_20]
target_vals   = [TARGET_MAE, TARGET_RMSE, TARGET_R2*100, 85, 92]
colors = [NORMAL_COLOR if a <= t else HYPO_COLOR
          for a, t in zip(achieved_vals[:3], target_vals[:3])]
colors += [NORMAL_COLOR, NORMAL_COLOR]
bars = ax8.barh(metric_names, achieved_vals, color=colors, alpha=0.85)
for bar, tv in zip(bars, target_vals):
    ax8.axvline(tv, color='gray', ls=':', lw=1)
ax8.set_title('Thesis Targets vs Achieved')
ax8.set_xlabel('Value')
for bar, val in zip(bars, achieved_vals):
    ax8.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', ha='left', fontsize=9)

plt.tight_layout()
dashboard_path = LOCAL_MODELS_DIR / 'metabolic_dashboard.png'
plt.savefig(dashboard_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"  ✅ Dashboard saved → {dashboard_path}")

### 6B — Individual plots (use in presentation / slides)

In [ ]:
# Helper: save individual plot
def save_plot(name):
    p = LOCAL_MODELS_DIR / f'metabolic_{name}.png'
    plt.savefig(p, dpi=150, bbox_inches='tight')
    print(f"  ✅ Saved: {p.name}")
    plt.show()

# ── 1. Training curves ────────────────────────────────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves — TwinPacemaker Metabolic LSTM', fontweight='bold')
a1.plot(history.history['mae'],     label='Train MAE', color='steelblue', lw=2)
a1.plot(history.history['val_mae'], label='Val MAE',   color='darkorange', lw=2)
a1.axhline(TARGET_MAE, color=HYPO_COLOR, ls='--', lw=1.5, label=f'Thesis Target {TARGET_MAE}')
a1.set(title='MAE per Epoch', xlabel='Epoch', ylabel='MAE (mg/dL)')
a1.legend()
a2.plot(history.history['loss'],     label='Train Loss', color='steelblue', lw=2)
a2.plot(history.history['val_loss'], label='Val Loss',   color='darkorange', lw=2)
a2.set(title='Loss per Epoch (Huber)', xlabel='Epoch', ylabel='Loss')
a2.legend()
save_plot('01_training_curves')

In [ ]:
# ── 2. Scatter: Predicted vs Actual ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_true_abs, y_pred_abs, s=3, alpha=0.2, color=PRED_COLOR)
ax.plot([35, 420], [35, 420], 'k--', lw=2, label='Perfect prediction')
ax.fill_between([35, 70], [35, 70], [35+420*0.15, 70+420*0.15], alpha=0.05, color=HYPO_COLOR)
for thresh, color, label in [(70, HYPO_COLOR, 'Hypoglycemia 70'), (180, HYPER_COLOR, 'Hyperglycemia 180')]:
    ax.axhline(thresh, color=color, ls=':', lw=1.5, label=label)
    ax.axvline(thresh, color=color, ls=':', lw=1.5)
ax.set(xlim=[35,420], ylim=[35,420],
       title=f'Predicted vs Actual Glucose\nR²={r2:.4f}  MAE={mae:.1f} mg/dL  RMSE={rmse:.1f} mg/dL',
       xlabel='Actual Glucose (mg/dL)', ylabel='Predicted Glucose (mg/dL)')
ax.legend(fontsize=10)
save_plot('02_scatter_pred_vs_actual')

In [ ]:
# ── 3. Error distribution ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(errors, bins=100, color=PRED_COLOR, edgecolor='white', alpha=0.85)
ax.axvline(0,    color='black',   lw=2,   label='Zero error')
ax.axvline(-15,  color=HYPO_COLOR, ls='--', lw=1.8, label='±15 mg/dL target')
ax.axvline(+15,  color=HYPO_COLOR, ls='--', lw=1.8)
ax.axvline(-20,  color=HYPER_COLOR, ls=':', lw=1.5, label='±20 mg/dL')
ax.axvline(+20,  color=HYPER_COLOR, ls=':', lw=1.5)
ax.set(title=f'Prediction Error Distribution  (Within ±15: {within_15:.1f}%  |  Within ±20: {within_20:.1f}%)',
       xlabel='Actual − Predicted (mg/dL)', ylabel='Frequency')
ax.legend()
save_plot('03_error_distribution')

In [ ]:
# ── 4. 60-min timeline ────────────────────────────────────────────────────────
N = min(600, len(y_true_abs))
t = np.arange(N) * 5
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(t, y_true_abs[:N], color=TRUE_COLOR, lw=1.8, label='Actual CGM', alpha=0.9)
ax.plot(t, y_pred_abs[:N], color=PRED_COLOR, lw=1.8, label='Predicted (60 min ahead)', alpha=0.9, ls='--')
ax.fill_between(t, 40, 70,  alpha=0.10, color=HYPO_COLOR,  label='Hypoglycemia (<70)')
ax.fill_between(t, 180, 420, alpha=0.10, color=HYPER_COLOR, label='Hyperglycemia (>180)')
ax.set(title='60-Minute Glucose Forecast — TwinPacemaker Digital Twin',
       xlabel='Time (minutes)', ylabel='Blood Glucose (mg/dL)',
       ylim=[40, 420])
ax.legend(ncol=2)
save_plot('04_timeline_forecast')

In [ ]:
# ── 5. Zone classification ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x_pos = np.arange(3)
labels = ['Hypoglycemia\n(<70 mg/dL)', 'Normal\n(70–180 mg/dL)', 'Hyperglycemia\n(>180 mg/dL)']
colors_zone = [HYPO_COLOR, NORMAL_COLOR, HYPER_COLOR]
axes[0].bar(x_pos - 0.2, true_counts, 0.38, label='Actual',    color=colors_zone, alpha=0.7, edgecolor='white')
axes[0].bar(x_pos + 0.2, pred_counts, 0.38, label='Predicted', color=colors_zone, alpha=1.0, edgecolor='black', lw=0.8)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(labels)
axes[0].set_title(f'Zone Classification (Accuracy={zone_acc:.1f}%)')
axes[0].set_ylabel('Sample Count')
axes[0].legend()

# Confusion-style F1 bars
event_names = ['Hypo\nPrecision', 'Hypo\nRecall', 'Hypo\nF1',
               'Hyper\nPrecision', 'Hyper\nRecall', 'Hyper\nF1']
event_vals  = [hypo_m['precision'], hypo_m['recall'], hypo_m['f1'],
               hyper_m['precision'], hyper_m['recall'], hyper_m['f1']]
e_colors    = [HYPO_COLOR]*3 + [HYPER_COLOR]*3
axes[1].bar(event_names, event_vals, color=e_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0.8, color='gray', ls='--', lw=1.2, label='Target 0.80')
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('Score')
axes[1].set_title('Alert Event Metrics (Clinical Safety)')
axes[1].legend()
for i, v in enumerate(event_vals):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
plt.tight_layout()
save_plot('05_zone_classification')

In [ ]:
# ── 6. Thesis metrics summary ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
metric_labels  = ['MAE (mg/dL)', 'RMSE (mg/dL)', 'R² × 100', 'Within ±15 (%)', 'Within ±20 (%)']
achieved       = [mae, rmse, r2*100, within_15, within_20]
targets        = [TARGET_MAE, TARGET_RMSE, TARGET_R2*100, 85, 92]
bar_colors     = [NORMAL_COLOR if (i >= 2 and achieved[i] >= targets[i]) or
                                  (i < 2 and achieved[i] <= targets[i])
                  else HYPO_COLOR for i in range(5)]

y_pos = np.arange(len(metric_labels))
ax.barh(y_pos, achieved, color=bar_colors, alpha=0.85, height=0.5, label='Achieved')
ax.plot(targets, y_pos, 'D', color='black', ms=8, zorder=5, label='Thesis Target')
ax.set_yticks(y_pos); ax.set_yticklabels(metric_labels)
ax.set_title('TwinPacemaker Metabolic LSTM — Thesis Performance Targets', fontweight='bold')
ax.set_xlabel('Value')
for i, (val, tgt) in enumerate(zip(achieved, targets)):
    ax.text(val + max(achieved)*0.01, i, f'{val:.2f}', va='center', fontsize=10)
ax.legend()
save_plot('06_thesis_targets_summary')

## 💾 Step 7 — Save models & sync to Google Drive → Windows

In [ ]:
# ─── Save model artifacts locally ─────────────────────────────────────────────
local_keras_path = LOCAL_MODELS_DIR / 'metabolic_stacked_lstm.keras'
model.save(local_keras_path)
print(f"  Model saved: {local_keras_path}")

# Save full model info JSON
model_info = {
    'model_name': 'TwinPacemaker_Metabolic_LSTM',
    'thesis_reference': 'Chapter 5 SF-07 / LF-2.2 — GRU Simulation Métabolique',
    'artifact': 'metabolic_stacked_lstm.keras',
    'architecture': {
        'type': 'Stacked LSTM',
        'layers': [128, 64, 32],
        'input_shape': [WINDOW_SIZE, FEATURE_DIM],
        'output': 'delta_glucose_scalar',
        'loss': 'huber',
        'total_params': int(model.count_params())
    },
    'dataset': {
        'name': 'GlucoBench',
        'kaggle': 'omenkj/glucobench-glucose-monitoring-and-lifestyle-data',
        'path': str(DATA_CSV),
        'feature_columns': FEATURE_COLS,
    },
    'normalization': norm_stats,
    'training': {
        'epochs_trained': len(history.history['loss']),
        'batch_size': BATCH_SIZE,
        'window_size': WINDOW_SIZE,
        'horizon_steps': HORIZON_STEPS,
        'horizon_minutes': HORIZON_STEPS * 5,
        'train_fraction': TRAIN_FRACTION,
        'seed': SEED,
    },
    'metrics': {
        'mae_mgdl': mae,
        'rmse_mgdl': rmse,
        'r2': r2,
        'mape_pct': mape,
        'rmse_gl': rmse_gl,
        'within_15_mgdl_pct': within_15,
        'within_20_mgdl_pct': within_20,
        'zone_accuracy_pct': zone_acc,
        'all_targets_met': all_met,
    },
    'alert_metrics': {
        'hypoglycemia': hypo_m,
        'hyperglycemia': hyper_m,
    },
    'generated_at_utc': datetime.now(timezone.utc).isoformat()
}

info_path = LOCAL_MODELS_DIR / 'metabolic_model_info.json'
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f"  Model info saved: {info_path}")

print(f"\n  Local artifacts in: {LOCAL_MODELS_DIR}")
for f in sorted(LOCAL_MODELS_DIR.iterdir()):
    print(f"    {f.name:45s}  {f.stat().st_size/1e3:8.1f} KB")

In [ ]:
# ─── Sync to Google Drive ─────────────────────────────────────────────────────
# This makes files available at:
#   D:\Vibe Coding\TwinPacemaker\models\metabolic\
# ... when you sync Drive to your Windows PC via Google Drive desktop client.

print("Syncing to Google Drive ...")
for src in sorted(LOCAL_MODELS_DIR.iterdir()):
    dst = DRIVE_MODELS_DIR / src.name
    shutil.copy2(src, dst)
    print(f"  ✅  Drive: {dst}")

print(f"\n🎯 All artifacts synced to Drive: {DRIVE_MODELS_DIR}")
print("\n📁 On Windows (after Drive sync):")
print("   D:\\Vibe Coding\\TwinPacemaker\\models\\metabolic\\")
for src in sorted(LOCAL_MODELS_DIR.iterdir()):
    print(f"     {src.name}")

## ✅ Final Summary

In [ ]:
print("")
print("═"*65)
print("   TwinPacemaker — Metabolic LSTM (SF-07 / LF-2.2)")
print("   FINAL RESULTS SUMMARY")
print("═"*65)
print(f"   Dataset       : GlucoBench (Kaggle)")
print(f"   Features      : {FEATURE_COLS}")
print(f"   Input shape   : ({WINDOW_SIZE}, {FEATURE_DIM}) — 10h lookback")
print(f"   Output        : delta_glucose → reconstructed 60-min abs")
print(f"   Parameters    : {model.count_params():,}")
print(f"   Epochs trained: {len(history.history['loss'])}")
print("─"*65)
print(f"   MAE      = {mae:.2f} mg/dL    target < {TARGET_MAE}  {'✅' if mae < TARGET_MAE else '❌'}")
print(f"   RMSE     = {rmse:.2f} mg/dL   target < {TARGET_RMSE}  {'✅' if rmse < TARGET_RMSE else '❌'}")
print(f"   R²       = {r2:.4f}        target > {TARGET_R2}  {'✅' if r2 > TARGET_R2 else '❌'}")
print(f"   RMSE g/L = {rmse_gl:.4f}       LF-2.2 bench: 0.12 g/L")
print(f"   ±15 mgdL = {within_15:.1f}%")
print(f"   ±20 mgdL = {within_20:.1f}%")
print(f"   Zone acc = {zone_acc:.1f}%")
print("─"*65)
print(f"   ALL TARGETS MET: {'✅ YES — Thesis-ready' if all_met else '⚠️  See metrics above'}")
print("═"*65)
print(f"   Artifacts saved to Drive → {DRIVE_MODELS_DIR}")